# Reading & Writing Data

## Spark's Unified I/O Model

Every read and write in Spark goes through the **DataFrameReader** (`spark.read`) and **DataFrameWriter** (`df.write`) APIs. These provide a consistent interface regardless of the underlying format or storage system.

### Supported Formats (built-in)

| Format | Extension | Best For |
|--------|-----------|----------|
| **Parquet** | `.parquet` | Analytical workloads — columnar, compressed, schema-embedded |
| **CSV** | `.csv` | Human-readable exchange, legacy systems |
| **JSON** | `.json` | Semi-structured data, REST API payloads |
| **ORC** | `.orc` | Hive/Hadoop ecosystems, similar benefits to Parquet |
| **Avro** | `.avro` | Row-based, great for streaming / Kafka |
| **Delta Lake** | (directory) | ACID transactions, time travel, schema evolution |
| **JDBC** | n/a | Relational databases (Postgres, MySQL, …) |

### Key Concepts Covered Here
- Writing a DataFrame to CSV, Parquet, and JSON
- Reading data back with `inferSchema` vs an explicit schema
- Understanding write **modes** (`overwrite`, `append`, `ignore`, `error`)
- Using `partitionBy` to organise data on disk and enable **predicate pushdown**

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("05 - Reading & Writing Data")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print(f"Spark {spark.version} ready.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 14:20:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3 ready.


In [2]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
import pyspark.sql.functions as F

# ── Build the source DataFrame ────────────────────────────────────────────────
employee_schema = StructType([
    StructField("emp_id",     IntegerType(), nullable=False),
    StructField("name",       StringType(),  nullable=False),
    StructField("department", StringType(),  nullable=False),
    StructField("location",   StringType(),  nullable=False),
    StructField("salary",     DoubleType(),  nullable=False),
])

employees_data = [
    (1, "Alice Chen",    "Engineering", "Baku",    95000.0),
    (2, "Bob Mammadov",  "Engineering", "Baku",    88000.0),
    (3, "Carol Smith",   "Marketing",   "Tbilisi", 72000.0),
    (4, "Dave Aliyev",   "Marketing",   "Baku",    68000.0),
    (5, "Eve Rustamova", "Finance",     "Baku",    81000.0),
    (6, "Frank Muller",  "Finance",     "Berlin",  79000.0),
    (7, "Grace Kim",     "Engineering", "Tbilisi", 91000.0),
    (8, "Hank Ibrahimov","HR",          "Baku",    62000.0),
    (9, "Iris Novak",    "HR",          "Warsaw",  60000.0),
    (10,"Jake Torres",   "Engineering", "Berlin",  98000.0),
]

emp_df = spark.createDataFrame(employees_data, schema=employee_schema)
emp_df.printSchema()
emp_df.show()

# ── Define output paths ───────────────────────────────────────────────────────
CSV_PATH     = "/home/jovyan/data/employees.csv"
PARQUET_PATH = "/home/jovyan/data/employees.parquet"
JSON_PATH    = "/home/jovyan/data/employees.json"

# ── Write as CSV ─────────────────────────────────────────────────────────────
# header=True: write column names in the first row
# mode="overwrite": replace any existing files at this path
emp_df.write.mode("overwrite").option("header", True).csv(CSV_PATH)
print(f"CSV written to {CSV_PATH}")

# ── Write as Parquet ──────────────────────────────────────────────────────────
# Parquet embeds the schema automatically — no extra options needed.
emp_df.write.mode("overwrite").parquet(PARQUET_PATH)
print(f"Parquet written to {PARQUET_PATH}")

# ── Write as JSON (newline-delimited JSON / JSONL) ────────────────────────────
emp_df.write.mode("overwrite").json(JSON_PATH)
print(f"JSON written to {JSON_PATH}")

root
 |-- emp_id: integer (nullable = false)
 |-- name: string (nullable = false)
 |-- department: string (nullable = false)
 |-- location: string (nullable = false)
 |-- salary: double (nullable = false)



+------+--------------+-----------+--------+-------+
|emp_id|          name| department|location| salary|
+------+--------------+-----------+--------+-------+
|     1|    Alice Chen|Engineering|    Baku|95000.0|
|     2|  Bob Mammadov|Engineering|    Baku|88000.0|
|     3|   Carol Smith|  Marketing| Tbilisi|72000.0|
|     4|   Dave Aliyev|  Marketing|    Baku|68000.0|
|     5| Eve Rustamova|    Finance|    Baku|81000.0|
|     6|  Frank Muller|    Finance|  Berlin|79000.0|
|     7|     Grace Kim|Engineering| Tbilisi|91000.0|
|     8|Hank Ibrahimov|         HR|    Baku|62000.0|
|     9|    Iris Novak|         HR|  Warsaw|60000.0|
|    10|   Jake Torres|Engineering|  Berlin|98000.0|
+------+--------------+-----------+--------+-------+

CSV written to /home/jovyan/data/employees.csv
Parquet written to /home/jovyan/data/employees.parquet
JSON written to /home/jovyan/data/employees.json


In [3]:
# ── Reading CSV: inferSchema vs explicit schema ───────────────────────────────
#
# inferSchema=True: Spark reads the file TWICE — first pass samples the data
# to guess types, second pass reads the actual records.
# This is convenient for exploration but slow on large files.

csv_inferred = (
    spark.read
    .option("header", True)       # first row is column names
    .option("inferSchema", True)  # let Spark guess types (costs an extra scan)
    .csv(CSV_PATH)
)

print("=== CSV with inferSchema ===")
csv_inferred.printSchema()  # types should match the original StructType
csv_inferred.show(5)

# ── Reading CSV with an explicit schema (preferred for production) ────────────
# Supplying the schema skips the sampling pass entirely — much faster.
csv_explicit = (
    spark.read
    .option("header", True)
    .schema(employee_schema)      # provide the schema we already defined above
    .csv(CSV_PATH)
)

print("=== CSV with explicit schema ===")
csv_explicit.printSchema()
csv_explicit.show(5)

=== CSV with inferSchema ===
root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- location: string (nullable = true)
 |-- salary: double (nullable = true)

+------+--------------+-----------+--------+-------+
|emp_id|          name| department|location| salary|
+------+--------------+-----------+--------+-------+
|     7|     Grace Kim|Engineering| Tbilisi|91000.0|
|     8|Hank Ibrahimov|         HR|    Baku|62000.0|
|     9|    Iris Novak|         HR|  Warsaw|60000.0|
|    10|   Jake Torres|Engineering|  Berlin|98000.0|
|     1|    Alice Chen|Engineering|    Baku|95000.0|
+------+--------------+-----------+--------+-------+
only showing top 5 rows

=== CSV with explicit schema ===
root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- location: string (nullable = true)
 |-- salary: double (nullable = true)

+------+--------------+-----------

In [5]:
# ── Reading Parquet ───────────────────────────────────────────────────────────
#
# Parquet files store the schema in a footer section of each file.
# Spark reads this footer before touching the data rows, so:
#   1. No inferSchema scan needed
#   2. Types are exact — no silent coercions (a problem with CSV)
#   3. Column pruning: if you only SELECT two columns, Spark reads only those
#      two column chunks from disk — the rest never leave storage.

parquet_df = spark.read.parquet(PARQUET_PATH)   # schema restored automatically

print("=== Parquet (schema auto-restored) ===")
parquet_df.printSchema()
parquet_df.show()

# Demonstrate column pruning: Spark will only read 'name' and 'salary' columns
print("=== Column pruning — only name + salary read from disk ===")
parquet_df.select("name", "salary").show()

=== Parquet (schema auto-restored) ===
root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- location: string (nullable = true)
 |-- salary: double (nullable = true)

+------+--------------+-----------+--------+-------+
|emp_id|          name| department|location| salary|
+------+--------------+-----------+--------+-------+
|     7|     Grace Kim|Engineering| Tbilisi|91000.0|
|     8|Hank Ibrahimov|         HR|    Baku|62000.0|
|     9|    Iris Novak|         HR|  Warsaw|60000.0|
|    10|   Jake Torres|Engineering|  Berlin|98000.0|
|     1|    Alice Chen|Engineering|    Baku|95000.0|
|     2|  Bob Mammadov|Engineering|    Baku|88000.0|
|     3|   Carol Smith|  Marketing| Tbilisi|72000.0|
|     4|   Dave Aliyev|  Marketing|    Baku|68000.0|
|     5| Eve Rustamova|    Finance|    Baku|81000.0|
|     6|  Frank Muller|    Finance|  Berlin|79000.0|
+------+--------------+-----------+--------+-------+

=== Column prunin

## Parquet vs CSV vs JSON — Trade-offs

### Parquet (recommended for analytics)

- **Columnar storage**: data is laid out column-by-column on disk. If a query touches 3 of 100 columns, only 3% of the raw bytes are read. Massive I/O savings for wide tables.
- **Built-in compression**: Parquet pairs column data with the best algorithm per column type (dictionary encoding, RLE, bit-packing). Typical compression ratio is **3–10×** over CSV.
- **Schema embedded**: the file is self-describing. No separate schema definition needed for readers.
- **Schema evolution**: you can safely add nullable columns without breaking existing readers.
- **Predicate pushdown**: Parquet stores row-group statistics (min/max per column per row group). Spark can skip entire row groups whose statistics prove no row matches the WHERE clause — without reading those rows.

### CSV

- **Human-readable** — open in any text editor or spreadsheet tool.
- **Universal** — every tool on earth can read/write CSV.
- **No schema** — all columns are read as strings unless you supply a schema or enable `inferSchema`.
- **Row-based** — even if you only need one column, every byte of the file must be scanned.
- **Fragile** — commas and newlines inside values require careful quoting and escaping.

### JSON (newline-delimited)

- **Self-describing at the row level** — each line carries its own field names, making it natural for semi-structured data with varying schemas.
- **Row-based** — same full-scan penalty as CSV.
- **Verbose** — keys are repeated for every row, so files are larger than CSV before any compression.
- **Nested structures** — natively supports arrays and nested objects, which CSV cannot represent without encoding tricks.

**Bottom line**: use Parquet (or Delta Lake) for everything in your Spark pipelines. Use CSV/JSON only at the ingestion boundary where external systems force that format.

In [6]:
# ── Write Modes ───────────────────────────────────────────────────────────────
#
# Spark supports four write modes that control what happens when the
# target path or table already exists:
#
#   "overwrite" — delete everything at the target path, then write fresh data.
#                 Most common in batch ETL where you rebuild the output each run.
#
#   "append"    — add new files alongside existing ones. Row counts accumulate.
#                 Use for incremental loads or streaming checkpoints.
#
#   "ignore"    — silently do nothing if the path already exists.
#                 Useful for idempotent pipelines: only write on first run.
#
#   "error"     — raise an exception if the path already exists (the DEFAULT).
#                 Protects against accidental overwrites in production.

demo_path = "/home/jovyan/data/mode_demo.parquet"

# First write — creates the files
emp_df.write.mode("overwrite").parquet(demo_path)
print(f"overwrite  -> row count: {spark.read.parquet(demo_path).count()}")

# Append — doubles the rows
emp_df.write.mode("append").parquet(demo_path)
print(f"append     -> row count: {spark.read.parquet(demo_path).count()}")

# Ignore — does nothing because the path exists
emp_df.limit(3).write.mode("ignore").parquet(demo_path)
print(f"ignore     -> row count: {spark.read.parquet(demo_path).count()}  (unchanged)")

# error mode demonstration (wrapped in try/except to keep the notebook running)
try:
    emp_df.write.mode("error").parquet(demo_path)   # default mode
except Exception as e:
    print(f"error mode -> raised: {type(e).__name__}: {str(e)[:80]}")

overwrite  -> row count: 10
append     -> row count: 20
ignore     -> row count: 20  (unchanged)
error mode -> raised: AnalysisException: [PATH_ALREADY_EXISTS] Path file:/home/jovyan/data/mode_demo.parquet already exis


In [7]:
import os

# ── partitionBy: organising data on disk for fast filtered reads ──────────────
#
# When you call .write.partitionBy("column"), Spark creates a separate
# subdirectory for each distinct value of that column:
#
#   /home/jovyan/data/employees_partitioned/
#       department=Engineering/
#           part-00000-....parquet
#       department=Finance/
#           part-00000-....parquet
#       ...
#
# When a query filters on 'department', Spark's file-listing step
# only opens the matching subdirectory — it never reads the other partitions.
# This is called PARTITION PRUNING and can reduce I/O by orders of magnitude
# on large datasets partitioned by date, country, or category.

PARTITIONED_PATH = "/home/jovyan/data/employees_partitioned.parquet"

# Write partitioned by 'department'
(
    emp_df
    .write
    .mode("overwrite")
    .partitionBy("department")   # one subdirectory per department
    .parquet(PARTITIONED_PATH)
)
print("Partitioned Parquet written.")

# Show the on-disk layout using os.walk so students can see the directory tree
print("\nOn-disk layout:")
for dirpath, dirnames, filenames in os.walk(PARTITIONED_PATH):
    depth = dirpath.replace(PARTITIONED_PATH, "").count(os.sep)
    indent = "    " * depth
    print(f"{indent}{os.path.basename(dirpath)}/")
    for fname in filenames:
        if not fname.startswith("."):   # skip hidden metadata files
            print(f"{indent}    {fname}")

# ── Predicate pushdown / partition pruning in action ─────────────────────────
# Spark reads ONLY the department=Engineering/ subdirectory here.
# Enable WARN logging or check the Spark UI -> SQL tab to confirm
# that only the matching partition files are opened.
engineering_only = (
    spark.read
    .parquet(PARTITIONED_PATH)
    .filter(F.col("department") == "Engineering")   # partition pruning kicks in
)

print("\nEngineering employees (only that partition was read):")
engineering_only.show()

# explain() confirms Spark pushed the filter down — look for
# 'PartitionFilters' in the physical plan output
print("\nPhysical plan (look for PartitionFilters):")
engineering_only.explain()

Partitioned Parquet written.

On-disk layout:
employees_partitioned.parquet/
    _SUCCESS
    department=Finance/
        part-00002-0f7d7d05-79f0-4424-a2b5-589e6ce1600c.c000.snappy.parquet
    department=Engineering/
        part-00000-0f7d7d05-79f0-4424-a2b5-589e6ce1600c.c000.snappy.parquet
        part-00003-0f7d7d05-79f0-4424-a2b5-589e6ce1600c.c000.snappy.parquet
    department=HR/
        part-00003-0f7d7d05-79f0-4424-a2b5-589e6ce1600c.c000.snappy.parquet
    department=Marketing/
        part-00001-0f7d7d05-79f0-4424-a2b5-589e6ce1600c.c000.snappy.parquet

Engineering employees (only that partition was read):
+------+------------+--------+-------+-----------+
|emp_id|        name|location| salary| department|
+------+------------+--------+-------+-----------+
|     1|  Alice Chen|    Baku|95000.0|Engineering|
|     2|Bob Mammadov|    Baku|88000.0|Engineering|
|     7|   Grace Kim| Tbilisi|91000.0|Engineering|
|    10| Jake Torres|  Berlin|98000.0|Engineering|
+------+------------+

In [ ]:
# Release cluster resources
spark.stop()
print("SparkSession stopped.")